In [ ]:
import yfinance as yf
import pandas as pd

# 1.1 Descargar datos históricos de Tesla
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")

# 1.2 Resetear el índice
tesla_data.reset_index(inplace=True)

# Mostrar las primeras 5 filas (Requisito de la rúbrica)
tesla_data.head()


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


In [2]:
import requests
from bs4 import BeautifulSoup

# 2.1 Descargar la página web con los ingresos de Tesla
url_tesla = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
html_data = requests.get(url_tesla).text

# 2.2 Analizar el HTML usando el objeto BeautifulSoup
soup = BeautifulSoup(html_data, "html.parser")

# Crear un DataFrame vacío para almacenar las filas
tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])

# Buscamos la tabla secundaria de ingresos anuales e iteramos por sus filas
for row in soup.find_all("tbody")[1].find_all("tr"):
    col = row.find_all("td")
    date = col[0].text
    revenue = col[1].text
    
    # Concatenamos de forma segura cada nueva fila
    new_row = pd.DataFrame({"Date": [date], "Revenue": [revenue]})
    tesla_revenue = pd.concat([tesla_revenue, new_row], ignore_index=True)

# 2.3 Limpieza obligatoria: eliminar comas y el símbolo de dólar ($)
tesla_revenue["Revenue"] = tesla_revenue['Revenue'].str.replace(r'[\$,]', '', regex=True)

# Eliminar posibles filas vacías o con valores nulos (NaN)
tesla_revenue.dropna(inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue['Revenue'] != ""]

# Mostrar las últimas 5 filas (Requisito estricto de la rúbrica)
tesla_revenue.tail()

,Date,Revenue
48,2010-09-30,31
49,2010-06-30,28
50,2010-03-31,21
52,2009-09-30,46
53,2009-06-30,27


In [3]:
# 3.1 Descargar los datos históricos de GameStop usando el ticker 'GME'
gamestop = yf.Ticker("GME")
gme_data = gamestop.history(period="max")

# 3.2 Resetear el índice para que la Fecha sea una columna accesible
gme_data.reset_index(inplace=True)

# Mostrar las primeras 5 filas (Requisito obligatorio de la rúbrica)
gme_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13 00:00:00-05:00,1.620128,1.693350,1.603296,1.691667,76216000,0.0,0.0
1,2002-02-14 00:00:00-05:00,1.712707,1.716074,1.670626,1.683251,11021600,0.0,0.0
2,2002-02-15 00:00:00-05:00,1.683250,1.687458,1.658002,1.674834,8389600,0.0,0.0
3,2002-02-19 00:00:00-05:00,1.666418,1.666418,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20 00:00:00-05:00,1.615921,1.662210,1.603296,1.662210,6892800,0.0,0.0


In [4]:
# 4.1 Descargar la página web con los ingresos de GameStop
url_gme = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
html_data_gme = requests.get(url_gme).text

# 4.2 Analizar el HTML con BeautifulSoup
soup_gme = BeautifulSoup(html_data_gme, "html.parser")

# Crear el DataFrame vacío para almacenar los datos
gme_revenue = pd.DataFrame(columns=["Date", "Revenue"])

# Buscamos la tabla de ingresos anuales (segundo elemento tbody) e iteramos
for row in soup_gme.find_all("tbody")[1].find_all("tr"):
    col = row.find_all("td")
    date = col[0].text
    revenue = col[1].text
    
    # Añadimos la fila de forma limpia
    new_row = pd.DataFrame({"Date": [date], "Revenue": [revenue]})
    gme_revenue = pd.concat([gme_revenue, new_row], ignore_index=True)

# 4.3 Limpieza obligatoria: eliminar comas y el signo de dólar ($)
gme_revenue["Revenue"] = gme_revenue['Revenue'].str.replace(r'[\$,]', '', regex=True)

# Eliminar filas vacías o nulas que puedan romper el gráfico posterior
gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue['Revenue'] != ""]

# Mostrar las últimas 5 filas (Requisito obligatorio de la rúbrica)
gme_revenue.tail()

,Date,Revenue
57,2006-01-31,1667
58,2005-10-31,534
59,2005-07-31,416
60,2005-04-30,475
61,2005-01-31,709


In [5]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def make_graph(stock_data, revenue_data, stock):
    # Crear un panel con 2 filas y 1 columna compartiendo el eje X (fechas)
    fig = make_subplots(
        rows=2, cols=1, 
        shared_xaxes=True, 
        subplot_titles=("Historical Share Price", "Historical Revenue"), 
        vertical_spacing = .3
    )
    
    # Filtrar datos históricos para una visualización óptima del periodo del ejercicio
    stock_data_specific = stock_data[stock_data.Date <= '2021-06-14']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-04-30']
    
    # Gráfico 1: Precio de la acción (Fila 1)
    fig.add_trace(go.Scatter(
        x=pd.to_datetime(stock_data_specific.Date), 
        y=stock_data_specific.Close.astype("float"), 
        name="Share Price"
    ), row=1, col=1)
    
    # Gráfico 2: Ingresos (Fila 2)
    fig.add_trace(go.Scatter(
        x=pd.to_datetime(revenue_data_specific.Date), 
        y=revenue_data_specific.Revenue.astype("float"), 
        name="Revenue"
    ), row=2, col=1)
    
    # Configurar títulos de los ejes
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($ Millions)", row=2, col=1)
    
    # Ajustes de diseño final
    fig.update_layout(
        showlegend=False, 
        height=600, 
        title=f"Dashboard interactivo para {stock}", 
        template="plotly_white"
    )
    
    fig.show()

In [6]:
make_graph(tesla_data, tesla_revenue, 'Tesla')

In [7]:
make_graph(gme_data, gme_revenue, 'GameStop')